In [ ]:
import os
import random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2
from sklearn.model_selection import StratifiedKFold
import timm  # Библиотека SOTA-моделей (включая трансформеры и ConvNeXt)

# -----------------------------------------------------------------------------
# 1. Воспроизводимость и Конфигурация
# -----------------------------------------------------------------------------
SEED = 42


def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True


seed_everything(SEED)
device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available() else "cpu"
)

# -----------------------------------------------------------------------------
# 2. Исправленный Dataset (с рабочей аугментацией)
# -----------------------------------------------------------------------------
train_transforms = v2.Compose(
    [
        v2.RandomRotation(degrees=12),
        v2.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.9, 1.1)),
    ]
)


class DigitDataset(Dataset):

    def __init__(self, X, y=None, transform=None):
        self.X = X
        self.y = y
        self.transform = transform  # ИСПРАВЛЕНО!

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        if self.transform:
            x = self.transform(x)
        if self.y is not None:
            return x, self.y[idx]
        return x


# -----------------------------------------------------------------------------
# 3. Модель 1: Наша кастомная CNN
# -----------------------------------------------------------------------------
class CustomCNN(nn.Module):

    def __init__(self, in_features=1, num_classes=10):
        super().__init__()

        def conv_block(in_f, out_f):
            return nn.Sequential(
                nn.Conv2d(in_f, out_f, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_f),
                nn.SiLU(),
                nn.Conv2d(out_f, out_f, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_f),
                nn.SiLU(),
                nn.MaxPool2d((2, 2)),
                nn.Dropout2d(0.15),
            )

        self.features = nn.Sequential(
            conv_block(in_features, 32),
            conv_block(32, 64),
            conv_block(64, 128),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# -----------------------------------------------------------------------------
# 4. Модель 2: Pretrained ConvNeXt из timm (Hugging Face экосистема)
# -----------------------------------------------------------------------------
class PretrainedTimmModel(nn.Module):

    def __init__(
        self, model_name="convnext_nano.in12k_ft_in1k", num_classes=10
    ):
        super().__init__()
        # Загружаем SOTA архитектуру с предобученными весами
        self.model = timm.create_model(
            model_name, pretrained=True, in_chans=1, num_classes=num_classes
        )

    def forward(self, x):
        return self.model(x)


# -----------------------------------------------------------------------------
# 5. Функция обучения и инференс с TTA
# -----------------------------------------------------------------------------
def train_model(
    model_cls,
    X_train_all,
    y_train_all,
    epochs=20,
    batch_size=64,
    lr=1e-3,
    n_splits=5,
):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    models = []

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X_train_all, y_train_all)
    ):
        train_sub = DigitDataset(
            X_train_all[train_idx],
            y_train_all[train_idx],
            transform=train_transforms,
        )
        val_sub = DigitDataset(X_train_all[val_idx], y_train_all[val_idx])

        train_loader = DataLoader(
            train_sub, batch_size=batch_size, shuffle=True
        )
        val_loader = DataLoader(
            val_sub, batch_size=batch_size * 2, shuffle=False
        )

        model = model_cls().to(device)
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=lr, weight_decay=1e-2
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=epochs
        )
        loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

        best_acc, best_weights = 0.0, None

        for epoch in range(epochs):
            model.train()
            for X, y in train_loader:
                X, y = X.to(device), y.to(device)
                optimizer.zero_grad()
                loss = loss_fn(model(X), y)
                loss.backward()
                optimizer.step()

            scheduler.step()

            # Валидация
            model.eval()
            correct, total = 0, 0
            with torch.inference_mode():
                for X, y in val_loader:
                    X, y = X.to(device), y.to(device)
                    preds = model(X).argmax(dim=1)
                    correct += (preds == y).sum().item()
                    total += len(y)

            val_acc = correct / total
            if val_acc > best_acc:
                best_acc = val_acc
                best_weights = model.state_dict().copy()

        model.load_state_dict(best_weights)
        models.append(model)
        print(f"--> Fold {fold + 1} Best Acc: {best_acc:.5f}")

    return models


# Функция TTA (Test-Time Augmentation)
def predict_tta(model, test_loader):
    model.eval()
    probs_list = []
    with torch.inference_mode():
        for X_batch in test_loader:
            X_orig = X_batch.to(device)
            # 1. Оригинальное фото
            p0 = F.softmax(model(X_orig), dim=1)

            # 2. Поворот на +5 градусов
            X_rot1 = v2.functional.rotate(X_batch, angle=5).to(device)
            p1 = F.softmax(model(X_rot1), dim=1)

            # 3. Поворот на -5 градусов
            X_rot2 = v2.functional.rotate(X_batch, angle=-5).to(device)
            p2 = F.softmax(model(X_rot2), dim=1)

            # Среднее по TTA
            probs_list.append(((p0 + p1 + p2) / 3.0).cpu().numpy())

    return np.vstack(probs_list)


# -----------------------------------------------------------------------------
# 6. Главный запуск Мульти-Модельного Ансамбля
# -----------------------------------------------------------------------------
# 1. Загрузка данных
train_df = pd.read_csv("/kaggle/input/competitions/digit-recognizer/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/digit-recognizer/test.csv")

X_raw = train_df.drop(columns=["label"]).values
y_raw = train_df["label"].values
X_test_raw = test_df.values

X_train_all = (
    torch.tensor(X_raw, dtype=torch.float32).reshape(-1, 1, 28, 28) / 255.0
)
y_train_all = torch.tensor(y_raw, dtype=torch.long)
X_test_all = (
    torch.tensor(X_test_raw, dtype=torch.float32).reshape(-1, 1, 28, 28)
    / 255.0
)

test_dataset = DigitDataset(X_test_all, transform=None)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# --- ШАГ A: Обучение 5 фолдов CustomCNN ---
print("\n=== [1/2] Обучение Семейства CustomCNN ===")
cnn_models = train_model(
    CustomCNN, X_train_all, y_train_all, epochs=25, lr=1e-3
)

# --- ШАГ B: Обучение 5 фолдов ConvNeXt ---
print("\n=== [2/2] Обучение Семейства Pretrained ConvNeXt ===")
convnext_models = train_model(
    PretrainedTimmModel, X_train_all, y_train_all, epochs=15, lr=5e-4
)

# --- ШАГ C: Генерация TTA-предсказаний и Блендинг ---
print("\nГенерация предсказаний с TTA и блендингом 10 моделей...")

final_probs = np.zeros((len(X_test_all), 10))

# Вклад 5 моделей CustomCNN (вес 50%)
for model in cnn_models:
    final_probs += (predict_tta(model, test_loader) / 5.0) * 0.5

# Вклад 5 моделей ConvNeXt (вес 50%)
for model in convnext_models:
    final_probs += (predict_tta(model, test_loader) / 5.0) * 0.5

# Выбор финальных меток
final_preds = final_probs.argmax(axis=1)

# Формирование сабмита
submission = pd.DataFrame(
    {"ImageId": range(1, len(final_preds) + 1), "Label": final_preds}
)
submission.to_csv("submission.csv", index=False)
print("Готово! Максимально жирный сабмит записан в submission.csv")